In [1]:
# Importing libraries and files
import pandas as pd
import numpy as np
import joblib
import time

from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import ParameterSampler
from xgboost import XGBClassifier

In [2]:
# Loading data and baseline model
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
REPORT_DIR = Path("../reports")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv")["disease_encoded"]
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["disease_encoded"]

label_encoder = joblib.load(DATA_DIR / "label_encoder.joblib")

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Number of classes:", len(label_encoder.classes_))

Training data: (80000, 174)
Testing data: (20000, 174)
Number of classes: 46


In [3]:
# Defining a managable parameter search
from sklearn.model_selection import ParameterSampler

param_distributions = {
    "n_estimators": [200, 300, 400],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

candidates = list(ParameterSampler(
    param_distributions,
    n_iter=6,
    random_state=42
))

print("Configurations to evaluate:", len(candidates))

Configurations to evaluate: 6


In [4]:
# Training and evaluating each configuration
from sklearn.model_selection import train_test_split

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

results = []

for i, params in enumerate(candidates, start=1):
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        **params
    )

    start = time.time()
    model.fit(X_fit, y_fit)
    elapsed = time.time() - start

    predictions = model.predict(X_val)

    results.append({
        **params,
        "macro_f1": f1_score(
            y_val, predictions,
            average="macro",
            zero_division=0
        ),
        "accuracy": accuracy_score(y_val, predictions),
        "training_seconds": elapsed
    })

    print(
        f"Configuration {i}/6 | "
        f"Macro F1: {results[-1]['macro_f1']:.4f} | "
        f"Accuracy: {results[-1]['accuracy']:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

results_df = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)

display(results_df)

ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [27]